# Day 27 In-Class Assignment: Baseline Modeling & Evaluation

**Use this notebook to guide your in-class work on Day 27.**

Building on Day 26, you will:
- Load your model-ready dataset from S3
- Define features and a target variable
- Split the data into train and test sets
- Train a baseline model
- Compute and interpret accuracy, precision, and recall
- Document one model limitation
- Update your modeling plan and commit everything to GitHub

**At the end of class, commit all changes to GitHub and submit this notebook to D2L.**

> **Name (FIRST and LAST):** _delete this text and type your name here_

---

## Learning objectives

By the end of this class period, you should be able to:

- **Explain** why held-out test data is necessary for honest evaluation
- **Implement** a train/test split using scikit-learn
- **Train** a baseline model in SageMaker
- **Compute and interpret** accuracy, precision, and recall
- **Identify and document** at least one meaningful limitation of the model
- **Commit** a clean, documented notebook to your GitHub repo

- - -

## Table of contents

1. [Setup and data loading](#setup)
2. [Define features and target](#features)
3. [Train/test split](#split)
4. [Encode categorical features](#encode)
5. [Train the baseline model](#train)
6. [Evaluate the model](#evaluate)
7. [Interpret and document](#interpret)
8. [Repo cleanup and Git commit](#git)
9. [Extended work (optional)](#extended)

- - -

<a id='setup'></a>

## 1. Setup and data loading 

For this assignment, you'll **create a new notebook in your GitHub repo** in your `notebooks` folder called `modeling_train_and_eval.ipynb` and use it to complete the steps below. You should make sure this notebook is running in the same SageMaker instance where you completed the data preparation steps on Day 26, so you can easily access the dataset you exported to S3 and don't need to worry about setting up a new instance.

Once you have the notebook **running in the SageMaker instance**, start by importing the libraries you'll need and loading the modeling dataset you exported from Athena on Day 26.

**To make all this work:**
- Make sure your AWS Learner Lab is running
- Confirm your SageMaker notebook instance is "InService"
- Open your cloned GitHub repo in the JupyterLab file browser (if for some reason it isn't there, you might need to clone it) and navigate to the `notebooks` folder
- Create a new notebook in that folder called `modeling_train_and_eval.ipynb`
- Copy, paste, and modify the code snippets below into cells in that notebook

**If your SageMaker instance or S3 path is not available**, let the an instructor know!

In [ ]:
# Import initial libraries
import pandas as pd
import numpy as np
import boto3
from io import StringIO
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# Load your modeling dataset from S3
# Replace the path below with your actual S3 path from Day 26
s3 = boto3.client('s3')
bucket_name = 'your-bucket-name' # replace with your bucket name (make sure you have the account regional suffix)
file_name = 'your-file-name.csv' # make sure to include the path if it's in a folder, e.g. 'modeling/modeling_data.csv'

obj = s3.get_object(Bucket=bucket_name, Key=file_name)
data = obj['Body'].read().decode('utf-8')
df = pd.read_csv(StringIO(data))

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Quick data check — confirm the shape, column names, and target variable
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

**Checkpoint:** Does the dataframe look like what you expected from Day 26?
- Correct number of rows and columns?
- Any unexpected missing values? Do you need to drop rows with missing values or impute them before modeling?
- Does the target column match what you documented in `notes/modeling_plan.md`?

Note any surprises or issues in the cell below before moving on.

> *Your notes here (double-click to edit):*



- - -

<a id='features'></a>

## 2. Define features and target 

Refer back to your `notes/modeling_plan.md` from Day 26. You documented:
- **Problem:** What you are trying to predict

- **Features:** The input columns
    - For the features, make sure to include only columns that would be available at prediction time, and exclude any columns that are proxies for the target or are otherwise potentially redundant

- **Target:** The output column (what the model should predict)

Use that plan to fill in the cell below once you've copied it into your new notebook. Make sure to update the `feature_cols` and `target_col` variables to match your dataset.

This example code also does a quick check on the class balance of the target variable, which is important to note before training a model. If your target is very imbalanced, you might want to consider techniques to address that (like resampling or using different evaluation metrics) before moving on to training. **You should have noted the class balance in your modeling plan on Day 26, so this is just a checkpoint to confirm that.**

<div class="alert alert-attention">

**Note:** Make sure your target variable is appropriate for your chosen model type. For example, if you want to train a simple logistic regression model, your target should be binary (two classes). For the example data table from the Day 26 ICA (the one for predicting high-volume agencies), if `volume_quartile` was your target, you'd need to binarize it — for example, `high_volume = (volume_quartile == 1).astype(int)` to predict only the highest-volume group.

</div>

In [ ]:
# Define feature columns and target column
# Replace these with your actual column names
feature_cols = ['col1', 'col2']  # <-- update this
target_col   = 'target'          # <-- update this

# If your target needs to be binarized or modified in some way, do it here
# Example: df['high_volume'] = (df['volume_quartile'] == 1).astype(int)

X = df[feature_cols]
y = df[target_col]

print(f"Features: {feature_cols}")
print(f"Target: {target_col}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}")

**Checkpoint:** Is your target roughly balanced between classes?

- If one class makes up more than ~80% of your data, your model might look accurate simply by predicting the majority class every time. This is worth noting as a potential limitation.
- Make a note of the class balance and what you could do to address it before moving on.

> *Your notes on class balance and you might address it when modeling the data:*



- - -

<a id='split'></a>

## 3. Train/test split 

At this point in your data science education, you should be more than familiar with the concept of splitting data into training and test sets. But as a reminder, we split the data into a **training set** (used to fit the model) and a **test set** (held out and only used to evaluate the trained model).

**Why does this matter?**  
A model that is evaluated on the same data it was trained on can appear to perform well simply because it memorized the training examples — not because it learned anything generalizable. Holding out a test set gives an honest picture of how the model will perform on new data. Again, **this shouldn't be news to you!**

**Important note**: In the example code below, we use `stratify=y` to ensure both splits have the same proportion of each class. You should make sure you think about whether this is appropriate for your data! The example also uses `random_state=42` so the split is reproducible.

In [ ]:
# Import train_test_split for splitting the data
from sklearn.model_selection import train_test_split

# Train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set:  {X_train.shape[0]} rows")
print(f"Test set:      {X_test.shape[0]} rows")
print(f"\nClass balance in training set:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nClass balance in test set:")
print(y_test.value_counts(normalize=True).round(3))

**Checkpoint:** Are the class proportions roughly the same in both the training set and the test set? Ideally, they should be (especially if you used `stratify=y`).

- - -

<a id='encode'></a>

## 4. Encode categorical features

Again, **hopefully you've encountered this before in one of your core data science courses**, but before training a model, we need every feature column to be **numeric**. Most machine learning algorithms work by multiplying feature values by learned coefficients, which requires numbers. Columns like `agency`, `borough`, and `problem` are strings and will cause a `ValueError` if passed directly to `model.fit()`.

**What is one-hot encoding?**  
One-hot encoding converts a categorical column into a set of binary (0/1) columns — one per unique category. For example, `borough = "BRONX"` becomes five columns: `borough_BRONX=1`, `borough_BROOKLYN=0`, `borough_MANHATTAN=0`, `borough_QUEENS=0`, `borough_STATEN ISLAND=0`. The model then learns a separate coefficient for each borough, capturing which boroughs are associated with the target.

**Why `ColumnTransformer`?**  
Our feature matrix typically has a mix of categorical columns (need encoding) and numeric columns (should pass through unchanged). `ColumnTransformer` lets us apply different transformations to different columns in a single, reproducible step.

<div class="alert alert-attention">

**Important — fit on training data only.**  
The transformer is **fit on `X_train` only**, then applied to both `X_train` and `X_test`. This mirrors real deployment: the encoder learns the category vocabulary from training data, and that same mapping is applied to new data. If you fit on the full dataset (including the test set), the encoder "sees" test-set categories during fitting — a form of **data leakage** that inflates apparent performance.

</div>

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Identify categorical vs. numeric columns automatically
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(include=['number']).columns.tolist()

print(f"Categorical columns ({len(cat_cols)}): {cat_cols}")
print(f"Numeric columns     ({len(num_cols)}): {num_cols}")

# Build the transformer:
#   - OneHotEncoder for categorical columns
#     drop='first'         avoids multicollinearity (k-1 dummies per feature)
#     sparse_output=False  returns a regular numpy array (easier to inspect)
#     handle_unknown='ignore'  silently zeros out categories not seen in training
#   - StandardScaler for numeric columns (optional but often helpful for models like Logistic Regression)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
        ('num', StandardScaler(), num_cols),  # scale numeric columns
    ]
)

# Fit ONLY on training data, then apply to both splits
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

# Recover feature names for later coefficient inspection
ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
encoded_feature_names = ohe_names + num_cols

print(f"\nEncoded feature matrix shape: {X_train_enc.shape}")
print(f"Total features after encoding: {len(encoded_feature_names)}")
print(f"\nFirst few encoded feature names: {encoded_feature_names[:10]}")

- - -

<a id='train'></a>

## 5. Train the baseline model 

Now it's time to train a baseline model!

For this part of the assignment you should **choose a model type that is appropriate for your target variable**. If you're doing binary classification, logistic regression is a good starting point. Or, even if you doing multi-class classification, logistic regression can still be a good baseline (it will just learn separate coefficients for each class). For continuous targets, linear or ridge regression is a common baseline.

When selecting a baseline model, it's good to intentionally start with a simple model like logistic regression for classification or linear/ridge regression for regression. This approach creates a simple, interpretable model that you can inspect and understand before moving on to more complex models later. It also gives you a performance floor to beat with more complex models later.

The code snippet below shows how to train a logistic regression model on the encoded training data. **You should adjust the model type and parameters as needed for your specific task.**

In [ ]:
# Import the logistic regression model
from sklearn.linear_model import LogisticRegression

# Train the logistic regression baseline (adjust the model type and parameters as needed for your specific task)
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_enc, y_train)

print("Model trained successfully.")
print(f"Classes: {model.classes_}")

In [ ]:
# Inspect the model coefficients
# After encoding, use encoded_feature_names instead of feature_cols
coef_df = pd.DataFrame({
    'feature': encoded_feature_names,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Model coefficients (top 10 by magnitude):")
print(coef_df.head(10).to_string(index=False))
print("\nPositive coefficient = feature pushes toward the positive class")
print("Negative coefficient = feature pushes toward the negative class")

- - -



## 6. Evaluate the model <a id='evaluate'></a>

Now we evaluate the trained model on the **test set** — data the model has never seen.

We'll compute three metrics:
- **Accuracy:** What fraction of predictions were correct overall?
- **Precision:** Of all the times the model predicted "positive", how often was it right?
- **Recall:** Of all the actual positive cases, how many did the model catch?

We'll also look at the **confusion matrix**, which shows exactly where the model made errors.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Generate predictions on the test set
y_pred = model.predict(X_test_enc)

# Compute metrics
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)

print(f"Accuracy:  {acc:.3f}")
print(f"Precision: {prec:.3f}")
print(f"Recall:    {rec:.3f}")

In [ ]:
# Plot the confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Confusion Matrix — Baseline Logistic Regression')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120)
plt.show()
print("Confusion matrix saved to confusion_matrix.png")

**Reading the confusion matrix:**

As a reminder, the confusion matrix for a binary classification problem is a 2x2 table that breaks down predictions into four categories:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

For your stakeholder question: which type of error is more costly — a false positive or a false negative? Think about this before moving on to the interpretation section.

**Important note**: Depending on your dataset and the balance of classes, you might want to also examine the **classification report**, which includes precision, recall, and F1-score for each class.

**If your classes are imbalanced, accuracy can be misleading (is it eerily close to the majority class proportion?), so it can be important to look at precision and recall, especially for the minority class!**

- - -



## 7. Interpret and document <a id='interpret'></a>

Getting metrics to print is only step one. The more important skill is being able to explain what those numbers mean — and what they *don't* mean.

**In the two cells below, write:**
1. A 2–3 sentence plain-language interpretation of your model's performance
2. One meaningful limitation of the model

These will also go into your `notes/modeling_plan.md` document in your repo, so make sure to write them clearly and thoughtfully!

### Interpretation

*Write 2–3 sentences here interpreting the model results in plain language. Aim to answer: How well does the model perform? Is precision or recall more important for your stakeholder? What does the confusion matrix suggest?*

> **Your interpretation:**



### (at least) One meaningful limitation

*Identify one (or more) reason(s) this model might be misleading or incomplete. Some possibilities:*
- *The target is imbalanced — the model may look accurate by predicting the majority class*
- *Features are too coarse (e.g., only agency and borough) to capture real variation*
- *The dataset covers a limited time period and may not generalize*
- *The baseline model is not ready for decision-making without further validation*
- *The derived target (`volume_quartile`) introduces errors from the original SQL logic*

> **Your limitation(s):**



### Update `notes/modeling_plan.md`

Open `notes/modeling_plan.md` in the JupyterLab file browser and add a new section with today's results. Here is a template:

```markdown
## Baseline Model Results

- **Model:** Logistic Regression (update if you used a different model type or parameters!)
- **Features used:** [list your feature_cols]
- **Target:** [target_col]
- **Train/test split:** 80/20, random_state=42, stratified (update if you used different parameters!)

### Metrics
- Accuracy:  [value]
- Precision: [value]
- Recall:    [value]

### Interpretation
[2-3 sentence plain-language summary]

### Limitation
[One meaningful limitation]
```

- - -



## 8. Repo cleanup and Git commit <a id='git'></a>

Before committing, take a moment to make sure your repo is tidy:

- Your `model_train_and_eval.ipynb` notebook is saved in `notebooks/` — not at the repo root
- `notes/modeling_plan.md` reflects the actual features, target, metrics, and limitation from today
- Any valuable saved figures (e.g., `confusion_matrix.png`) are in a sensible location

Then commit and push with a descriptive message.

**Checkpoint:** After pushing, verify on GitHub.com that your commit appears and that `notebooks/modeling_train_and_eval.ipynb` and the updated `notes/modeling_plan.md` are visible.

- - -



## 9. Extended work (optional) <a id='extended'></a>

If you finish the core assignment early, try one or more of the following. These are not required, but they deepen the modeling practice and give you more material for your portfolio.

---

### Option A: Add a second baseline model

Train a decision tree or random forest and compare its metrics to logistic regression.

In [ ]:
# Option A: Decision tree baseline for comparison
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model.fit(X_train_enc, y_train)
y_pred_tree = tree_model.predict(X_test_enc)

print("Decision Tree — Test set performance")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_tree):.3f}")
print(f"  Precision: {precision_score(y_test, y_pred_tree, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_test, y_pred_tree, zero_division=0):.3f}")
print()
print("Logistic Regression — Test set performance (for comparison)")
print(f"  Accuracy:  {acc:.3f}")
print(f"  Precision: {prec:.3f}")
print(f"  Recall:    {rec:.3f}")

---

### Option B: Tune one hyperparameter

The `C` parameter in logistic regression controls regularization strength. Try a few values and see if the test accuracy changes meaningfully.

In [ ]:
# Option B: Try different values of C
for C_val in [0.01, 0.1, 1.0, 10.0, 100.0]:
    m = LogisticRegression(C=C_val, max_iter=1000, random_state=42)
    m.fit(X_train_enc, y_train)
    preds = m.predict(X_test_enc)
    print(f"C={C_val:6.2f}  |  Accuracy: {accuracy_score(y_test, preds):.3f}  "
          f"Precision: {precision_score(y_test, preds, zero_division=0):.3f}  "
          f"Recall: {recall_score(y_test, preds, zero_division=0):.3f}")

*Does changing `C` make a meaningful difference for your dataset? Why or why not? Write a brief note here.*



---

### Option C: Check for class imbalance

A common failure mode for classification models is predicting the majority class almost every time — which can look like good accuracy but is not useful. Check whether this is happening with your model.

In [ ]:
# Option C: Compare model recall vs. a majority-class baseline
majority_class = y_train.value_counts().idxmax()
y_pred_majority = [majority_class] * len(y_test)

print(f"Majority class: {majority_class}")
print(f"Majority-class baseline accuracy: {accuracy_score(y_test, y_pred_majority):.3f}")
print(f"Your model accuracy:              {acc:.3f}")
print()
print("If your model barely beats the majority-class baseline, it may not be learning much.")

---

### Option D: Draft a stakeholder sentence

In plain language (no jargon, no metric names), write one or two sentences that explain what this model says about the original business question. Could you say this to a non-technical manager?

> *Your stakeholder-facing summary:*



- - -

### Congratulations, you're done!

Submit this assignment by uploading it to the course Desire2Learn web page. Go to the In-class assignments section, find the appropriate submission link, and upload it there.

See you in class!

---
© Copyright 2026, The Department of Computational Mathematics, Science and Engineering at Michigan State University.